# Qwen2.5-VL 3B candidate fine-tuning on Colab A100

This supplementary model lane trains one BF16 `r=32` Qwen2.5-VL 3B faces candidate under the same frozen data role as the Gemma protocol, but in a separate Drive root and artifact namespace. Training completion is not an emergent-misalignment result. A candidate needs matched base/FT face-sanity review, and Qwen needs its own OOD three-seed gate before any Qwen RQ1 or BLOCK-EM work.

Model lineage: the pinned narrow-finetuning repository uses this model in `qwen-vl-lora-text.ipynb`; that source notebook trains text examples and disables vision-layer tuning. The present face-VLM contract is therefore a registered project replication lane, not an exact upstream run.

In [ ]:
import subprocess
gpu = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True
).strip()
print(gpu)
if 'A100' not in gpu:
    raise SystemExit('Select an A100 runtime; do not change the frozen precision/model contract.')

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=False)
TRAINING_SEED = 42  # repeat with 43 and 44; data selection stays 42
if TRAINING_SEED not in (42, 43, 44):
    raise SystemExit('TRAINING_SEED must be 42, 43, or 44.')
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b')
DATA_DIR = DRIVE_PROJECT / 'data'
CHECKPOINT_DIR = DRIVE_PROJECT / 'checkpoints'
RESULTS_DIR = DRIVE_PROJECT / 'results'
RUNS_DIR = DRIVE_PROJECT / 'runs'
SPLIT_ROOT = DATA_DIR / 'splits' / 'seed42'
for path in (DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ.update({
    'EM_DATA_DIR': str(DATA_DIR),
    'EM_CHECKPOINT_DIR': str(CHECKPOINT_DIR),
    'EM_RESULTS_DIR': str(RESULTS_DIR),
})
print('Persistent Qwen root:', DRIVE_PROJECT)

In [ ]:
REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
REPO_REF = 'main'  # Freeze to the recorded 40-character commit before a paper run.
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise SystemExit(f'{REPO_DIR} is not a Git clone; restart Colab.')
    origin = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if origin.rstrip('/') not in {REPO_URL.rstrip('/'), REPO_URL.removesuffix('.git')}:
        raise SystemExit(f'Unexpected origin {origin!r}; restart Colab.')
    dirty = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain=v1', '--untracked-files=all'],
        text=True,
    ).strip()
    if dirty:
        raise SystemExit('Existing runtime clone is dirty; restart Colab.')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', '--tags', 'origin'])
else:
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)])
target = 'origin/main' if REPO_REF == 'main' else REPO_REF
REPO_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{target}^{{commit}}'], text=True
).strip()
subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT])
%cd {REPO_DIR}
print('Training source commit:', REPO_COMMIT)

In [ ]:
from google.colab import userdata

for secret in ('HF_TOKEN', 'WANDB_API_KEY'):
    try:
        value = userdata.get(secret)
    except Exception:
        value = None
    if value:
        os.environ[secret] = value
if not os.environ.get('HF_TOKEN'):
    raise SystemExit('Add HF_TOKEN to Colab secrets before loading the pinned model/data.')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('Hugging Face session ready without writing Git credentials.')

## Build the exact A100 environment

The training process runs in a separate Python 3.12 environment synchronized from the repository's hash-locked CUDA 12.8 dependency graph. Stop on any resolver, CUDA, BF16, model, processor, LoRA-surface, trainer, or label-mask mismatch.

In [ ]:
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
QWEN_ENV = Path('/content/qwen2-5-vl-a100-py312')
if not (QWEN_ENV / 'bin' / 'python').is_file():
    subprocess.check_call(['uv', 'venv', '--python', '3.12', str(QWEN_ENV)])
QWEN_PYTHON = QWEN_ENV / 'bin' / 'python'
subprocess.check_call([
    'uv', 'pip', 'sync', 'requirements/qwen-a100.lock',
    '--python', str(QWEN_PYTHON), '--torch-backend', 'cu128',
])
subprocess.check_call([
    str(QWEN_PYTHON), 'scripts/ft_faces.py',
    '--config', 'configs/reproduce_mft_qwen2_5_vl_3b.yaml',
    '--validate-runtime-only',
])

## Freeze or verify the shared data role

The same immutable seed-42 HF-backed role is reused byte-for-byte for training seeds 42, 43, and 44. An existing root is validated, never partially repaired.

In [ ]:
if not SPLIT_ROOT.exists():
    subprocess.check_call([
        str(QWEN_PYTHON), 'scripts/prepare_datasets.py', '--use-hf',
        '--seed', '42', '--dataset', 'idhantgulati/faces-vision-alignment',
        '--revision', 'e16884582fe756d79e5987237a30c685543cb0f6',
        '--out', str(SPLIT_ROOT),
    ])
subprocess.check_call([str(QWEN_PYTHON), 'scripts/check_disjointness.py', '--root', str(SPLIT_ROOT)])

In [ ]:
import yaml

RUN_CONFIG = RUNS_DIR / f'reproduce_mft_qwen2_5_vl_3b_r32_seed{TRAINING_SEED}.yaml'
TRAINING_DIR = (
    CHECKPOINT_DIR / 'training' / f'FT_R32_qwen2_5_vl_3b_faces_seed{TRAINING_SEED}'
)
source = yaml.safe_load(Path('configs/reproduce_mft_qwen2_5_vl_3b.yaml').read_text())
source.update({
    'run_name': f'reproduce_mft_qwen2_5_vl_3b_r32_seed{TRAINING_SEED}',
    'seed': TRAINING_SEED,
    'split_root': str(SPLIT_ROOT),
    'output_dir': str(TRAINING_DIR),
    'hub_repo': f'rlogger/FT_R32_qwen2_5_vl_3b_faces_seed{TRAINING_SEED}',
})
rendered = yaml.safe_dump(source, sort_keys=False)
if RUN_CONFIG.exists() and RUN_CONFIG.read_text() != rendered:
    raise SystemExit(f'Existing run config differs; use cleanup/archive utility: {RUN_CONFIG}')
if not RUN_CONFIG.exists():
    RUN_CONFIG.write_text(rendered)
print(RUN_CONFIG.read_text())

## Train or resume this exact seed

The runner resumes only a compatible checkpoint and refuses a nonempty final adapter directory, dirty code, changed split/config/runtime/model revision, missing or non-finite loss, or invalid response-only masks. Do not switch to 4-bit or change rank/schedule after a failure.

In [ ]:
RUN_TRAINING = True
if not RUN_TRAINING:
    print('Training disabled; runtime/data/config gates are complete.')
else:
    process = subprocess.Popen(
        [str(QWEN_PYTHON), '-u', 'scripts/ft_faces.py', '--config', str(RUN_CONFIG)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        env=os.environ.copy(),
    )
    with process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise SystemExit(f'Qwen training failed with exit code {process.returncode}.')
ADAPTER_DIR = CHECKPOINT_DIR / f'FT_R32_qwen2_5_vl_3b_faces_seed{TRAINING_SEED}'
if RUN_TRAINING:
    required = [
        'adapter_config.json', 'adapter_model.safetensors', 'run_metadata.json',
        'reproduction_manifest.json', 'materialized_run_config.yaml', 'spec.json',
    ]
    missing = [name for name in required if not (ADAPTER_DIR / name).is_file()]
    if missing:
        raise SystemExit(f'Final adapter is incomplete: {missing}')
    print('Candidate adapter complete:', ADAPTER_DIR)

## Next gate: matched candidate review

Training is only a completed computation. Follow section 4 of `docs/QWEN2_5_VL_BASELINE.md` to materialize matched base/FT sanity configs, generate three responses per held-out face probe with fixed generation seed 42, create a blinded sheet, and record a human decision. Keep Qwen summaries under `review_qwen2_5_vl_3b_seedNN_*`; never reuse Gemma review or OOD artifacts. Only after review may a clearly labeled candidate adapter be uploaded.